# H09 — The catalog as a population of events

**Part VII · the laboratory of lecture 17: Part A is shown in class, Part B is done at home**

**Data.**

- `labs/data/gwtc_events.csv`: the table of the confident events of the public catalogs of the LIGO, Virgo and KAGRA Collaborations, from GWTC-1 through GWTC-5.0. It has 391 rows and 18 columns: one row for each event. `load_catalog` reads it. Its name is in `labs/config.toml`, the configuration of the laboratories.
- The table comes from the event API of the Gravitational Wave Open Science Center (GWOSC). It was taken once on 15 November 2026. It is committed with the course, so everything runs offline. `labs/data/make_gwtc_events.py` rebuilds it. An event that is in more than one release is kept once, in the most recent release.
- The columns are the name of the event, its release, the GPS time, the masses and their error bars, the luminosity distance, the redshift, the effective spin `chi_eff`, the network SNR, the false alarm rate `far` and $P_{\mathrm{astro}}$. The masses are in the source frame and in solar masses. The distance is in megaparsecs.
- The catalog papers count 390 events with $P_{\mathrm{astro}} \ge 0.5$ through GWTC-5.0 (LIGO Scientific, Virgo and KAGRA Collaborations 2026). The extra row is GW230630_070659: GWOSC lists it but marks it as having evidence of instrumental origin. The rows of O4a carry the release name GWTC-4.1. This is the update of GWTC-4.0 published with GWTC-5.0: it counts 229 events through O4a, where GWTC-4.0 counted 218 (LIGO Scientific, Virgo and KAGRA Collaborations 2026; LIGO Scientific, Virgo and KAGRA Collaborations 2026).
- For 109 of the 391 rows, all from O4, the catalog gives no source properties: the masses, the distance and the redshift are empty.
- Nothing in the notebook is simulated.

**Libraries.**

- `pandas`: reads the table, counts the events of each release, drops the empty rows and computes medians and quantiles.
- `numpy`: arrays, logarithms, histograms, random resampling and the masks.
- `matplotlib.pyplot`: all the plots.
- `tomllib`: reads `labs/config.toml`.
- `os` and `sys`: find the folder of the course, so that the course module can be imported.
- `gwcourse_palette`: the course colors.

**What we do.**

- Part A, the demonstration: the table, the events against time, the detection rate of each run, the mass plane, the uncertainty on the chirp mass and on the total mass, the selection effect and a claim about the effective spin, tested with its uncertainty.
- Part B, at home: the rate of three periods, the events outside the two clusters of the mass plane, the best and the worst measured chirp masses, the distance of light and heavy binaries against the horizon. Then a claim of your own.

> **References**
>
> (LIGO Scientific, Virgo and KAGRA Collaborations 2026), (LIGO Scientific, Virgo and KAGRA Collaborations 2026), (LIGO Scientific, Virgo and KAGRA Collaborations 2026), (LIGO Scientific Collaboration and Virgo Collaboration 2016), (LIGO Scientific Collaboration and Virgo Collaboration 2017), (LIGO Scientific, Virgo and KAGRA Collaborations 2025)

## Tasks

**Part A. The demonstration.** We show how the plots and the numbers of lecture 17 are made from the table. All the code is written out and runs cell by cell. Under each cell we say what the result shows. Every section that uses the masses, the distance or the spin drops the empty rows with `select_events` and prints how many rows it kept.

- **A1.** The table: the columns and the rows of each release, with and without masses.
- **A2.** The events against time: the function `gps_to_year`, the cumulative number and the observing runs.
- **A3.** The detection rate of each observing run: the functions `run_years` and `assign_runs`.
- **A4.** The mass plane: the function `plot_mass_plane`, two clusters, GW150914, GW170817 and the heaviest primary.
- **A5.** The uncertainty on the chirp mass and on the total mass: the function `fractional_error`, for all the events and for three classes of total mass.
- **A6.** The selection effect: the network SNR against the distance, with the total mass as color. The function `reach_distance`.
- **A7.** A claim to test: the effective spin is centered on zero. The functions `bootstrap_interval` and `fraction_positive`.

**Part B. Your turn.** The functions of Part A are given. We use them with our own choices.

- **B1.** The detection rate of O1 + O2, O3 and O4. The factor between the first and the last.
- **B2.** The mass plane: the events in neither cluster, the gap between 3 and 5 solar masses and the heaviest primaries.
- **B3.** The events with the best and the worst measured chirp mass: what do they have in common?
- **B4.** The distance of light, middle and heavy binaries against the horizon of lecture 17.
- **B5.** A claim to test about the primary masses, with a figure and a number with its uncertainty.

In [ ]:
# libraries
import os
import sys
import tomllib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# the course root holds gwcourse_palette.py: find it from wherever we are
here = os.path.abspath('.')
while not os.path.exists(os.path.join(here, 'gwcourse_palette.py')):
    if os.path.dirname(here) == here:
        raise FileNotFoundError('gwcourse_palette.py not found: open the notebook from the course folder')
    here = os.path.dirname(here)
sys.path.insert(0, here)

# the configuration of the laboratories: the folder of the data and the file of the catalog
with open(os.path.join(here, 'labs', 'config.toml'), 'rb') as config_file:
    config = tomllib.load(config_file)
DATA_DIR = os.path.join(here, 'labs', config['data']['dir'])
CATALOG_FILE = os.path.join(DATA_DIR, config['catalog']['file'])

# the course colors
from gwcourse_palette import P, apply_style
apply_style()

## A1. The table

The catalog of lecture 17 is a table with one row for each confident event. `load_catalog(path)` reads it from the file named in the configuration.

`select_events(events, columns, verbose=True)` keeps the rows that have a value in each of the `columns` and prints how many it kept. We need it because 109 rows have no source properties.

We print the columns and the number of rows of each release, with and without the masses.

In [ ]:
def load_catalog(path):
    """Read the table of the catalog: one row for each event.

    Args:
        path: the csv file of the catalog.

    Returns:
        events: a pandas table, one row for each event and one column for each quantity.
    """
    events = pd.read_csv(path)
    return events


def select_events(events, columns, verbose=True):
    """Keep the events that have a value in every one of the columns.

    Args:
        events: the table of the catalog.
        columns: the names of the columns that must not be empty, a list.
        verbose: print how many rows were kept (default True).

    Returns:
        kept: the rows of the table that have all the columns.
    """
    kept = events.dropna(subset=columns)
    if verbose:
        print(f"{len(kept)} of {len(events)} events have: {', '.join(columns)}")
    return kept


# the table: how many rows and which columns
events = load_catalog(CATALOG_FILE)
print(f"{len(events)} rows, {events.shape[1]} columns")
print(', '.join(events.columns))

# for each release, oldest first: the rows, the rows that have the masses and the rows that have not
masses_known = events['mass_1_source'].notna()
print(f"\n{'release':20s} {'rows':>5s} {'with masses':>12s} {'without':>8s}")
releases = ['GWTC-1-confident', 'GWTC-2.1-confident', 'GWTC-3-confident', 'GWTC-4.1', 'GWTC-5.0']
for release in releases:
    in_release = events['catalog.shortName'] == release
    n_rows = in_release.sum()
    n_masses = (in_release & masses_known).sum()
    print(f"{release:20s} {n_rows:5d} {n_masses:12d} {n_rows - n_masses:8d}")
print(f"{'all':20s} {len(events):5d} {masses_known.sum():12d} {(~masses_known).sum():8d}")

**What the result shows.** The table has 391 rows and 18 columns. The release of an event is the most recent one in which it appears. So GWTC-1 keeps one row, GW170817: GWTC-2.1 reanalyzed the other events of O1 and O2.

| release | rows | with masses | without |
|---|---|---|---|
| GWTC-1 | 1 | 1 | 0 |
| GWTC-2.1 | 54 | 54 | 0 |
| GWTC-3 | 35 | 35 | 0 |
| GWTC-4.1 | 140 | 88 | 52 |
| GWTC-5.0 | 161 | 104 | 57 |
| all | 391 | 282 | 109 |

All the 109 rows without masses are from O4. For these events the catalog gives no source properties, so the sections that follow use 282 rows at most. The 90 rows of the first three releases and the 140 of GWTC-4.1 are the 229 events through O4a of the catalog papers and the flagged row GW230630_070659. The 161 rows of GWTC-5.0 are the new events.

## A2. The events against time

The `GPS` column is the arrival time of each event, in seconds from 6 January 1980, with no leap seconds. `gps_to_year(gps, first_year=1980, offset_days=5)` turns it into a calendar year. The GPS zero is 5 days into 1980 and a year of 365.25 days has 31 557 600 s.

`date_to_gps(date, leap_seconds=18)` does the opposite for a date: it is right to within a few seconds. We use it to draw the observing runs. The dictionary `RUNS` holds the start and the end of each run, as the catalog papers give them (LIGO Scientific Collaboration and Virgo Collaboration 2019; LIGO Scientific and Virgo Collaborations 2021; LIGO Scientific, Virgo and KAGRA Collaborations 2023; LIGO Scientific, Virgo and KAGRA Collaborations 2026; LIGO Scientific, Virgo and KAGRA Collaborations 2026).

We plot the cumulative number of events: one step for each event. The runs are shaded.

In [ ]:
def gps_to_year(gps, first_year=1980, offset_days=5):
    """Convert GPS times to the calendar year, to within a day.

    GPS time counts seconds from 6 January 1980 and skips no leap seconds.

    Args:
        gps: the GPS times (s), a number or a column of the table.
        first_year: the year of the GPS zero (default 1980).
        offset_days: the days of that year before the GPS zero (default 5).

    Returns:
        year: the calendar year as a number, for example 2015.7.
    """
    # the seconds since the start of the first year
    seconds = gps + offset_days * 86400
    # a year of 365.25 days has 31557600 s
    year = first_year + seconds / 31557600.0
    return year


def date_to_gps(date, leap_seconds=18):
    """Convert a date to the GPS time, to within a few seconds.

    Args:
        date: the date as text, for example '2019-04-01 15:00' (UTC).
        leap_seconds: the leap seconds added to UTC since the GPS zero (s, default 18).

    Returns:
        gps: the GPS time (s).
    """
    # the seconds from the GPS zero, 6 January 1980, then the leap seconds
    seconds = (pd.Timestamp(date) - pd.Timestamp('1980-01-06')).total_seconds()
    gps = seconds + leap_seconds
    return gps


# the observing runs: the (start, end) of each one, as the catalog papers give them
RUNS = {'O1': ('2015-09-12', '2016-01-19'),
        'O2': ('2016-11-30', '2017-08-25'),
        'O3a': ('2019-04-01 15:00', '2019-10-01 15:00'),
        'O3b': ('2019-11-01 15:00', '2020-03-27 17:00'),
        'O4a': ('2023-05-24 15:00', '2024-01-16 16:00'),
        'O4b': ('2024-04-10 15:00', '2025-01-28 17:00')}

# the calendar year of each event, in time order, and the number of events up to each one
year = np.sort(gps_to_year(events['GPS']).to_numpy())
number = np.arange(1, year.size + 1)

# the cumulative number: one step for each event, with the observing runs shaded
fig, ax = plt.subplots(figsize=(9, 4.5))
for name, (start, end) in RUNS.items():
    ax.axvspan(gps_to_year(date_to_gps(start)), gps_to_year(date_to_gps(end)),
               color=P['GREY_LIGHT'], lw=0)
ax.step(year, number, where='post', color=P['BLUE'], lw=2)
ax.set_xlabel('year')
ax.set_ylabel('cumulative number of events')

# the numbers of the figure in a box, in the empty corner
n_o4 = np.sum(year >= 2023.0)
summary = (f"{year.size} events, {year.min():.1f} to {year.max():.1f}\n"
           f"{n_o4} of them in 2023 to 2025 (O4)\n"
           f"grey: the observing runs")
ax.text(0.03, 0.95, summary, transform=ax.transAxes, ha='left', va='top', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='white', edgecolor=P['GREY_MID']))
plt.tight_layout()
plt.show()

# the events found in each calendar year
for calendar_year in range(2015, 2026):
    n_year = np.sum((year >= calendar_year) & (year < calendar_year + 1))
    print(f"{calendar_year}: {n_year:3d} events")

**What the result shows.** The curve rises only inside the shaded stretches and is flat between them. The steps get steeper from run to run. The events found in each calendar year are 3 in 2015, 8 in 2017, 58 in 2019, 21 in 2020, 136 in 2023, 149 in 2024 and 16 in 2025. There are none in 2016, 2018, 2021 and 2022. In the three years 2023 to 2025, O4, the detectors found 301 of the 391 events.

## A3. The detection rate of each run

A flat stretch is time when the detectors were off. To compare the runs we divide the events of each one by the time the detectors observed. This is the detection rate $N/T_{\mathrm{obs}}$, in events a year of observation. It is not the merger rate density $\mathcal{R} = N/\langle VT\rangle$ of lecture 17: that one divides by the sensitive volume–time and describes the sources, not the detectors.

`run_years(start, end)` gives the duration of one run in years. `assign_runs(gps, run_ends)` gives each event the number of the first run that ends after it. Seven events were found in the engineering time just before O4a or O4b: they count in the run that follows.

We print the events, the years and the rate of each run and plot the rates.

In [ ]:
def run_years(start, end):
    """Give the duration of one observing run.

    Args:
        start: the start of the run, as text, for example '2019-04-01 15:00'.
        end: the end of the run, as text.

    Returns:
        years: the duration (years of 365.25 days).
    """
    seconds = (pd.Timestamp(end) - pd.Timestamp(start)).total_seconds()
    years = seconds / 86400.0 / 365.25
    return years


def assign_runs(gps, run_ends):
    """Give each event the number of the first run that ends after it.

    An event found just before a run, in the engineering time that precedes it,
    goes to that run.

    Args:
        gps: the GPS times of the events (s), an array.
        run_ends: the GPS times of the ends of the runs (s), in time order.

    Returns:
        run_number: for each event, the position in run_ends of the first end after it.
    """
    run_number = np.searchsorted(run_ends, gps)
    return run_number


# each event goes to a run, then the rate: the events found divided by the years of observation
names = list(RUNS)
run_ends = [date_to_gps(RUNS[name][1]) for name in names]
run_number = assign_runs(events['GPS'].to_numpy(), run_ends=run_ends)
rates = []
print(f"{'run':5s} {'events':>6s} {'years':>6s} {'events a year':>14s}")
for position, name in enumerate(names):
    n_events = int(np.sum(run_number == position))
    years = run_years(*RUNS[name])
    rates.append(n_events / years)
    print(f"{name:5s} {n_events:6d} {years:6.2f} {rates[position]:14.0f}")

# the rate of each run as a bar
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(names, rates, color=P['BLUE'])
ax.bar_label(bars, fmt='%.0f')
ax.set_ylabel('events a year of observation')
ax.set_ylim(0, 1.15 * max(rates))

# the numbers of the figure in a box
summary = (f"O2: {rates[1]:.0f} a year\n"
           f"O3: {rates[2]:.0f} and {rates[3]:.0f}\n"
           f"O4: {rates[4]:.0f} and {rates[5]:.0f}\n"
           f"O4a over O2: {rates[4] / rates[1]:.0f} times")
ax.text(0.03, 0.95, summary, transform=ax.transAxes, ha='left', va='top', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='white', edgecolor=P['GREY_MID']))
plt.tight_layout()
plt.show()

**What the result shows.** The rate grows from run to run.

| run | events | years | events a year |
|---|---|---|---|
| O1 | 3 | 0.35 | 8 |
| O2 | 8 | 0.73 | 11 |
| O3a | 44 | 0.50 | 88 |
| O3b | 35 | 0.40 | 87 |
| O4a | 140 | 0.65 | 216 |
| O4b | 161 | 0.80 | 201 |

O4a is about 20 times O2. Inside O3 and inside O4 the rate does not grow: it is 88 and 87, then 216 and 201. The sky did not change between the runs: the instruments did. A detector that sees twice as far finds 8 times as many events, because the volume goes as the cube of the distance. A factor 20 in the count needs a factor $20^{1/3} = 2.7$ in the distance.

## A4. The mass plane

Each event has a heavier mass $m_{1}$ and a lighter mass $m_{2}$, in the source frame and in solar masses. `plot_mass_plane(m1, m2, ax=None, color=None, label=None, size=25)` plots $m_{2}$ against $m_{1}$ on logarithmic axes. The dotted line is $m_{2} = m_{1}$.

We use the events that have both masses. We count two groups: both masses below 3 solar masses and both above 5. We mark GW150914, GW170817 and the event with the heaviest primary.

In [ ]:
def plot_mass_plane(m1, m2, ax=None, color=None, label=None, size=25):
    """Plot the lighter mass against the heavier one on log axes.

    The dotted line is m2 = m1, the equal mass binaries.

    Args:
        m1: the heavier masses (solar masses).
        m2: the lighter masses (solar masses).
        ax: the axes to draw on (default None: the current axes).
        color: the color of the points (default the blue of the course).
        label: the label of the points in a legend (default None).
        size: the area of a point (default 25).

    Returns:
        ax: the axes that were drawn on.
    """
    if ax is None:
        ax = plt.gca()
    if color is None:
        color = P['BLUE_MID']
    ax.loglog(m1, m2, 'o', color=color, alpha=0.7, markersize=np.sqrt(size), label=label)
    # the equal mass line, from the lightest to the heaviest mass of the plot
    line = np.array([1.0, 200.0])
    ax.plot(line, line, color=P['GREY_MID'], lw=1.0, ls=':')
    ax.set_xlabel(r'$m_1$ ($M_\odot$)')
    ax.set_ylabel(r'$m_2$ ($M_\odot$)')
    return ax


# the events that have both masses
masses = select_events(events, columns=['mass_1_source', 'mass_2_source'])
m1 = masses['mass_1_source']
m2 = masses['mass_2_source']

# the two clusters: both masses below 3, and both above 5; the others are in neither
neutron = (m1 < 3.0) & (m2 < 3.0)
black = (m1 > 5.0) & (m2 > 5.0)
print(f"both masses below 3 solar masses: {neutron.sum()} events, "
      f"median {m1[neutron].median():.2f} and {m2[neutron].median():.2f}")
print(f"both masses above 5 solar masses: {black.sum()} events, "
      f"median {m1[black].median():.1f} and {m2[black].median():.1f}")
print(f"in neither: {len(masses) - neutron.sum() - black.sum()} events")

# the plane, with three events marked: the offsets keep the names off the points
fig, ax = plt.subplots(figsize=(6, 6))
plot_mass_plane(m1, m2, ax=ax)
marked = (('GW150914', (50, -80)), ('GW170817', (30, 30)), ('GW231123_135430', (-90, -10)))
for name, offset in marked:
    row = masses[masses['commonName'] == name].iloc[0]
    ax.plot(row['mass_1_source'], row['mass_2_source'], 'o', color=P['BRICK'], markersize=7)
    ax.annotate(name[:8], (row['mass_1_source'], row['mass_2_source']), xytext=offset,
                textcoords='offset points', fontsize=10, color=P['BRICK'],
                arrowprops=dict(arrowstyle='-', color=P['BRICK']))

# the numbers of the figure in a box, in the empty corner above the equal mass line
summary = (f"{len(masses)} events\n"
           f"{neutron.sum()} with both masses below 3\n"
           f"{black.sum()} with both above 5\n"
           f"{len(masses) - neutron.sum() - black.sum()} in neither")
ax.text(0.04, 0.96, summary, transform=ax.transAxes, ha='left', va='top', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='white', edgecolor=P['GREY_MID']))
plt.tight_layout()
plt.show()

# the event with the heaviest primary
heaviest = m1.idxmax()
print(masses.loc[heaviest, ['commonName', 'mass_1_source', 'mass_2_source']].to_string())

**What the result shows.** The table has both masses for 282 events.

| group | events | median $m_{1}$ | median $m_{2}$ |
|---|---|---|---|
| both masses below 3 $M_\odot$ | 2 | 1.78 | 1.29 |
| both masses above 5 $M_\odot$ | 273 | 35.6 | 24.2 |
| neither | 7 | | |

The two events of the first group are GW170817 and GW190425. Their masses are those of neutron stars. The 273 events of the second group have masses of black holes: the cloud runs from 5 to about 100 solar masses and lies close to the dotted line, so the two masses are of the same order. The 7 events in neither group have one mass below 3 solar masses and a heavier partner: B2 lists them. Between 3 and 5 solar masses the plane is almost empty.

The event with the heaviest primary is GW231123, with 137 and 101 solar masses. Pair instability should leave no black hole between a lower edge of about 40 to 65 solar masses and an upper edge near 130 solar masses (Woosley 2017; Farmer et al. 2019; Woosley and Heger 2021). The lower edge is uncertain because it depends on the rate of one nuclear reaction. At least one of the two black holes of GW231123 is probably inside that gap (LIGO Scientific, Virgo and KAGRA Collaborations 2025).

## A5. The uncertainty on the chirp mass and on the total mass

The catalog gives each mass as the median of its posterior, with a lower and an upper error bar that bound the 90 percent credible interval. The lower bar is negative. The fractional uncertainty is half the width of the interval divided by the median. `fractional_error(value, lower, upper)` computes it.

We use the chirp mass $\mathcal{M} = (m_{1}m_{2})^{3/5}/(m_{1}+m_{2})^{1/5}$ and the total mass $M = m_{1}+m_{2}$. We plot their distributions. We compute the fractional uncertainty of both for each event and take the median over all the events and over three classes of total mass.

In [ ]:
def fractional_error(value, lower, upper):
    """Compute the fractional uncertainty from the error bars of the catalog.

    Args:
        value: the median of the quantity (same unit as the error bars).
        lower: the lower error bar, a negative number.
        upper: the upper error bar, a positive number.

    Returns:
        fraction: half the width of the 90 percent credible interval, divided by the median.
    """
    # the lower error bar is stored as a negative number, so the width is upper minus lower
    width = upper - lower
    fraction = width / 2.0 / value
    return fraction


# the events that have the chirp mass, the total mass and their error bars
columns = ['chirp_mass_source', 'chirp_mass_source_lower', 'chirp_mass_source_upper',
           'total_mass_source', 'total_mass_source_lower', 'total_mass_source_upper']
both = select_events(events, columns=columns)

# the fractional uncertainty of the chirp mass and of the total mass of each event
frac_chirp = fractional_error(both['chirp_mass_source'], both['chirp_mass_source_lower'],
                              both['chirp_mass_source_upper'])
frac_total = fractional_error(both['total_mass_source'], both['total_mass_source_lower'],
                              both['total_mass_source_upper'])
print(f"median fractional error: chirp mass {100 * frac_chirp.median():.0f} %, "
      f"total mass {100 * frac_total.median():.0f} %")
print(f"the chirp mass is the better measured in {100 * np.mean(frac_chirp < frac_total):.0f} % of the events")

# the same medians for light, middle and heavy binaries
classes = ((0.0, 20.0), (20.0, 60.0), (60.0, 1000.0))
median_chirp = []
median_total = []
print(f"\n{'total mass':>12s} {'events':>6s} {'chirp':>6s} {'total':>6s}")
for low, high in classes:
    pick = (both['total_mass_source'] >= low) & (both['total_mass_source'] < high)
    median_chirp.append(100 * frac_chirp[pick].median())
    median_total.append(100 * frac_total[pick].median())
    print(f"{low:4.0f} to {high:4.0f} {pick.sum():6d} {median_chirp[-1]:5.0f}% {median_total[-1]:5.0f}%")

# left: the two distributions; right: the medians of the fractional error for the three classes
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].hist(events['chirp_mass_source'].dropna(), bins=30, color=P['SAGE'], alpha=0.85, label='chirp mass')
axes[0].hist(events['total_mass_source'].dropna(), bins=30, color=P['BRICK'], alpha=0.55, label='total mass')
axes[0].set_xlabel(r'source frame mass ($M_\odot$)')
axes[0].set_ylabel('number of events')
axes[0].legend(frameon=False, loc='upper right')
positions = np.arange(len(classes))
axes[1].bar(positions - 0.2, median_chirp, width=0.4, color=P['SAGE'], label='chirp mass')
axes[1].bar(positions + 0.2, median_total, width=0.4, color=P['BRICK'], alpha=0.7, label='total mass')
axes[1].set_xticks(positions)
axes[1].set_xticklabels(['below 20', '20 to 60', 'above 60'])
axes[1].set_xlabel(r'total mass ($M_\odot$)')
axes[1].set_ylabel('median fractional error (%)')
axes[1].set_ylim(0, 30)
axes[1].legend(frameon=False, loc='upper left', ncol=2)

# the numbers of the figure in a box, in the empty corner of the left plot
summary = (f"median error\n"
           f"chirp mass {100 * frac_chirp.median():.0f} %\n"
           f"total mass {100 * frac_total.median():.0f} %\n"
           f"chirp mass better\nin {100 * np.mean(frac_chirp < frac_total):.0f} % of the events")
axes[0].text(0.97, 0.72, summary, transform=axes[0].transAxes, ha='right', va='top', fontsize=10,
             bbox=dict(boxstyle='round', facecolor='white', edgecolor=P['GREY_MID']))
plt.tight_layout()
plt.show()

**What the result shows.** 281 events have both masses and their error bars. The median fractional error is 14 % on the chirp mass and 16 % on the total mass. The chirp mass is the better measured in only 43 % of the events. The classes show where the difference is.

| total mass | events | chirp mass | total mass |
|---|---|---|---|
| below 20 $M_\odot$ | 39 | 5 % | 10 % |
| 20 to 60 $M_\odot$ | 102 | 10 % | 14 % |
| above 60 $M_\odot$ | 140 | 20 % | 18 % |

For a light binary the chirp mass is measured twice as well as the total mass. Above 60 solar masses the order is reversed. The frequency evolution of the inspiral depends at leading order on the chirp mass, so the inspiral determines this combination. A light binary stays in the band for many cycles and the chirp mass is known to a few percent. A heavy binary leaves the band after a few cycles of inspiral. Most of its signal is merger and ringdown, which depend on the total mass, so the advantage of the chirp mass is lost. The mass ratio enters the phase later and is measured much less well in both cases.

## A6. The selection effect

The events of a catalog are those that were loud enough. We plot the network SNR $\varrho$ against the luminosity distance $D_{L}$, with the total mass as color. We use the events that have a distance and a total mass.

The SNR falls as $1/D_{L}$. So an event with SNR $\varrho$ at the distance $D_{L}$ would be seen up to $D_{L}\,\varrho/\varrho_{\min}$, where $\varrho_{\min}$ is the smallest SNR that is seen. `reach_distance(distance, snr, snr_min=8.0)` computes this. We take for $\varrho_{\min}$ the 5th percentile of the SNR of the 391 events. We apply it to GW170817 and compare it with the ten heaviest events.

In [ ]:
def reach_distance(distance, snr, snr_min=8.0):
    """Estimate the distance to which an event could be seen.

    The SNR falls as 1 / distance, so the event is seen up to distance * snr / snr_min.

    Args:
        distance: the luminosity distance of the event (Mpc).
        snr: the network SNR of the event.
        snr_min: the smallest SNR that is seen (default 8).

    Returns:
        reach: the distance at which the SNR falls to snr_min (Mpc).
    """
    reach = distance * snr / snr_min
    return reach


# the events with a distance and a total mass, and the smallest SNR of the catalog
known = select_events(events, columns=['luminosity_distance', 'total_mass_source'])
rho_min = events['network_matched_filter_snr'].quantile(0.05)

# the network SNR against the luminosity distance, the total mass as color
fig, ax = plt.subplots(figsize=(8, 4.8))
points = ax.scatter(known['luminosity_distance'], known['network_matched_filter_snr'],
                    c=known['total_mass_source'], cmap='viridis', s=30)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xticks([200, 500, 1000, 2000, 5000])
ax.set_xticklabels(['200', '500', '1000', '2000', '5000'])
ax.set_yticks([6, 8, 10, 20, 40, 80])
ax.set_yticklabels(['6', '8', '10', '20', '40', '80'])
ax.set_xlabel(r'luminosity distance $D_L$ (Mpc)')
ax.set_ylabel(r'network SNR $\varrho$')
plt.colorbar(points, ax=ax, label=r'total mass ($M_\odot$)')

# GW170817: its distance, its SNR and the reach that follows
bns = events[events['commonName'] == 'GW170817'].iloc[0]
reach_bns = reach_distance(bns['luminosity_distance'], bns['network_matched_filter_snr'], snr_min=rho_min)

# the ten heaviest events: their masses and their distances
heavy = known.nlargest(10, 'total_mass_source')
farther = heavy['luminosity_distance'].median() / reach_bns

# the numbers of the figure in a box, in the empty corner
summary = (f"5 % of the events have SNR below {rho_min:.1f}\n"
           f"GW170817: {bns['luminosity_distance']:.0f} Mpc, SNR {bns['network_matched_filter_snr']:.0f}\n"
           f"seen to {reach_bns:.0f} Mpc\n"
           f"ten heaviest: median {heavy['luminosity_distance'].median():.0f} Mpc")
ax.text(0.97, 0.97, summary, transform=ax.transAxes, ha='right', va='top', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='white', edgecolor=P['GREY_MID']))
plt.tight_layout()
plt.show()

# the ten heaviest events and the comparison with GW170817
print(f"95 % of the events have a network SNR above {rho_min:.1f}")
print(f"GW170817: {bns['luminosity_distance']:.0f} Mpc at SNR {bns['network_matched_filter_snr']:.1f}, "
      f"so seen to about {reach_bns:.0f} Mpc")
print(f"the ten heaviest: total mass {heavy['total_mass_source'].min():.0f} to "
      f"{heavy['total_mass_source'].max():.0f}, distance {heavy['luminosity_distance'].min():.0f} to "
      f"{heavy['luminosity_distance'].max():.0f} Mpc, median {heavy['luminosity_distance'].median():.0f} Mpc")
print(f"farthest event in the table: {known['luminosity_distance'].max():.0f} Mpc")
print(f"the heavy ones are {farther:.0f} times farther, in a volume {farther ** 3:.1e} times larger")

**What the result shows.** 281 events have a distance and a total mass. The region of large distance and low mass is empty: the far events are all heavy. The SNR of 95 % of the events is above 7.8, which we take as $\varrho_{\min}$.

| | distance | note |
|---|---|---|
| GW170817 | 40 Mpc | SNR 33.0, seen to 168 Mpc |
| the ten heaviest events | 2200 to 7300 Mpc | median 4490 Mpc, total mass 116 to 236 $M_\odot$ |
| the farthest event | 8280 Mpc | |

The heavy binaries are seen 27 times farther than a neutron star binary, in a volume $1.9\times10^{4}$ times larger. A heavier binary radiates more and is visible farther away. This describes the instrument and not the universe: a plot of the catalog is not a map of the sources. A statement about the population must be corrected for the selection function $\langle VT\rangle(\theta)$ of lecture 17.

## A7. A claim to test

We test one statement against the table: *the effective spin is centered on zero*. The effective spin $\chi_{\mathrm{eff}}$ is the combination of the spins along the orbital angular momentum that the inspiral measures best. Each event gives one number, the median of its posterior.

We state the claim as numbers: the median of $\chi_{\mathrm{eff}}$ and the fraction of the events with $\chi_{\mathrm{eff}} > 0$, each with a 90 percent interval. `bootstrap_interval(values, statistic=np.median, n_boot=2000, level=90.0, seed=1)` draws `n_boot` samples of the same size as the data, with replacement, computes the statistic on each and returns the interval that holds `level` percent of the results. `fraction_positive(values)` gives the fraction of values above zero.

In [ ]:
def bootstrap_interval(values, statistic=np.median, n_boot=2000, level=90.0, seed=1):
    """Find the interval of a statistic by resampling the events with replacement.

    Args:
        values: the values, one for each event, an array.
        statistic: the function that gives the number to measure (default the median).
        n_boot: the number of resamples (default 2000).
        level: the probability inside the interval (percent, default 90).
        seed: the seed of the random numbers (default 1).

    Returns:
        low: the lower end of the interval.
        high: the upper end of the interval.
    """
    rng = np.random.default_rng(seed)
    results = np.zeros(n_boot)
    # each resample has as many events as the data, drawn with replacement
    for i in range(n_boot):
        resample = rng.choice(values, size=values.size, replace=True)
        results[i] = statistic(resample)
    # the tails that are left out on each side
    tail = (100.0 - level) / 2.0
    low = np.percentile(results, tail)
    high = np.percentile(results, 100.0 - tail)
    return low, high


def fraction_positive(values):
    """Give the fraction of the values that are above zero.

    Args:
        values: the values, an array.

    Returns:
        fraction: the number of positive values divided by the number of values.
    """
    fraction = np.mean(values > 0.0)
    return fraction


# the claim: the effective spin is centered on zero; the events that have a value
spins = select_events(events, columns=['chi_eff'])['chi_eff'].to_numpy()

# the median, the share of positive values and the 90 % interval of each, by bootstrap
median_spin = np.median(spins)
low_median, high_median = bootstrap_interval(spins, statistic=np.median)
share = fraction_positive(spins)
low_share, high_share = bootstrap_interval(spins, statistic=fraction_positive)
print(f"median chi_eff {median_spin:.3f}, 90 % interval {low_median:.3f} to {high_median:.3f}")
print(f"{100 * share:.0f} % of the events have chi_eff above zero, 90 % interval "
      f"{100 * low_share:.0f} to {100 * high_share:.0f} %")
print(f"the values spread with a standard deviation of {spins.std():.3f}")

# the distribution, with zero, the median and its interval
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.hist(spins, bins=30, color=P['SAGE'], alpha=0.85)
ax.axvspan(low_median, high_median, color=P['BRICK'], alpha=0.35, lw=0)
ax.axvline(median_spin, color=P['BRICK'], lw=1.5)
ax.axvline(0.0, color=P['GREY_DARK'], lw=1.2, ls='--')
ax.set_xlabel(r'effective spin $\chi_{\mathrm{eff}}$')
ax.set_ylabel('number of events')

# the numbers of the figure in a box; dashed: zero, red: the median and its interval
summary = (f"{spins.size} events\n"
           f"median {median_spin:.2f}, interval {low_median:.2f} to {high_median:.2f}\n"
           f"{100 * share:.0f} % above zero ({100 * low_share:.0f} to {100 * high_share:.0f} %)\n"
           f"dashed: zero; red: the median and its interval")
ax.text(0.97, 0.97, summary, transform=ax.transAxes, ha='right', va='top', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='white', edgecolor=P['GREY_MID']))
plt.tight_layout()
plt.show()

**What the result shows.** 273 events have $\chi_{\mathrm{eff}}$. The median is 0.05, with a 90 percent interval from 0.04 to 0.06. 66 % of the events have $\chi_{\mathrm{eff}}$ above zero, with an interval from 61 to 71 %. The values spread with a standard deviation of 0.136.

The claim as the table supports it: *$\chi_{\mathrm{eff}}$ is small and more often positive. The median is 0.05, small against the spread of 0.136. Two events out of three are above zero.* The interval of the median does not include zero.

Two reasons why it could be wrong. First, each value is the median of a posterior with a width of its own. The table does not carry it: the bootstrap treats the medians as exact, so the interval is too narrow. Second, selection: a binary with spins aligned with the orbit is louder and is seen farther, so the catalog can have a median above the one of the population.

## Part B. Your turn: the rate, the masses, the selection and a claim

Part A showed how the table is read. In Part B we ask our own questions of it. These are the functions we have defined for you. They are the functions of Part A. Call them with keywords, as in Part A.

- `load_catalog(path)` reads the table of the catalog from the file `path`.
- `select_events(events, columns, verbose=True)` returns the rows that have a value in every one of `columns` and prints how many it kept.
- `gps_to_year(gps, first_year=1980, offset_days=5)` returns the calendar year of a GPS time.
- `date_to_gps(date, leap_seconds=18)` returns the GPS time of a date given as text.
- `run_years(start, end)` returns the duration of one run in years, from two dates given as text.
- `assign_runs(gps, run_ends)` returns for each event the position, in `run_ends`, of the first run that ends after it.
- `plot_mass_plane(m1, m2, ax=None, color=None, label=None, size=25)` plots $m_{2}$ against $m_{1}$ on log axes.
- `fractional_error(value, lower, upper)` returns half the width of the 90 percent interval divided by the median.
- `reach_distance(distance, snr, snr_min=8.0)` returns the distance in megaparsecs to which an event could be seen.
- `bootstrap_interval(values, statistic=np.median, n_boot=2000, level=90.0, seed=1)` returns the lower and the upper end of the interval of a statistic.
- `fraction_positive(values)` returns the fraction of the values that are above zero.

**What we have from Part A.** `events`, `RUNS`, `names`, `run_number`, `masses`, `neutron`, `black`, `both`, `frac_chirp`, `frac_total`, `known`, `rho_min` and `spins`.

### B1. The rate of three periods

The catalog papers group the runs as O1 + O2, O3 and O4. Compute the detection rate of each group: the events found divided by the years of observation. By what factor is the rate of O4 higher than the rate of O1 + O2?

*Hint:* `names` is the list of the six runs and `run_number` gives for each event the position of its run in it, from 0 for O1 to 5 for O4b. The events of a group are those with `run_number` between the first and the last position of the group. The years are the sum of `run_years(*RUNS[name])` over the runs of the group.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### B2. The events in neither cluster

In A4 seven events were in neither cluster. List them with their two masses. Which one has the lightest companion? How many events have a component between 3 and 5 solar masses? List the three heaviest primaries and find out why the heaviest is unusual.

*Hint:* the rows in neither group are `masses[~(neutron | black)]`. A column is between 3 and 5 with `masses['mass_1_source'].between(3.0, 5.0)`. The three heaviest rows are `masses.nlargest(3, 'mass_1_source')`.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### B3. The best and the worst measured chirp mass

Find the five events with the smallest and the five with the largest fractional uncertainty on the chirp mass. For each print the name, the total mass, the network SNR and the two fractional uncertainties. What do the five best have in common? And the five worst?

*Hint:* `both` holds the rows that have the chirp mass and the total mass with their error bars. `frac_chirp` and `frac_total` are their fractional errors, in the same order. Add them as columns with `both.assign(frac_chirp=frac_chirp, frac_total=frac_total)`, then use `nsmallest` and `nlargest`.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### B4. Light and heavy binaries against the horizon

Split the events of `known` into three classes of total mass: below 30, from 30 to 60 and above 60 solar masses. For each class print the number of events and the median distance, chirp mass and network SNR. Divide the median distance of the heavy class by that of the light class. Compare it with the ratio of the horizons of lecture 17, $r_{\mathrm{h}} \propto \mathcal{M}^{5/6}$, computed from the median chirp masses of the two classes.

*Hint:* a class is `known[(known['total_mass_source'] >= low) & (known['total_mass_source'] < high)]`. The ratio of the horizons is `(chirp_heavy / chirp_light) ** (5.0 / 6.0)`.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### B5. A claim to test (optional)

Test one statement about the primary masses: *there are more events with $m_{1}$ between 30 and 40 solar masses than in the bins beside it.* Count the events of the table in bins 10 solar masses wide from 10 to 60. Compare the bin from 30 to 40 with the mean of the two bins beside it. The error of a count $n$ is $\sqrt{n}$. Write the excess as a number of standard deviations. Draw the histogram with the error bars. Then write the claim in one sentence: the figure, the number with its uncertainty and one reason why it could be wrong.

*Hint:* `np.histogram(masses['mass_1_source'], bins=edges)` with `edges = np.arange(10.0, 70.0, 10.0)`. The error of the excess $n_{2} - (n_{1}+n_{3})/2$ is $\sqrt{n_{2} + (n_{1}+n_{3})/4}$.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


## Software

`numpy`, `pandas` and `matplotlib`, on the event table in `labs/data/gwtc_events.csv`, built by `labs/data/make_gwtc_events.py`. The catalog is (LIGO Scientific, Virgo and KAGRA Collaborations 2026; LIGO Scientific, Virgo and KAGRA Collaborations 2026). Nothing in the notebook is simulated.

**References**

- LIGO Scientific, Virgo and KAGRA Collaborations (2026). *GWTC-5.0: Observations from the Second Part of the Fourth LIGO-Virgo-KAGRA Observing Run and Updates to the Gravitational-Wave Transient Catalog*.
- LIGO Scientific, Virgo and KAGRA Collaborations (2026). *GWTC-4.0: Updating the Gravitational-Wave Transient Catalog with Observations from the First Part of the Fourth LIGO-Virgo-KAGRA Observing Run*.
- LIGO Scientific, Virgo and KAGRA Collaborations (2026). *GWTC-5.0: Population Properties of Merging Compact Binaries*.
- LIGO Scientific Collaboration and Virgo Collaboration (2016). *Observation of Gravitational Waves from a Binary Black Hole Merger*.
- LIGO Scientific Collaboration and Virgo Collaboration (2017). *GW170817: Observation of Gravitational Waves from a Binary Neutron Star Inspiral*.
- LIGO Scientific, Virgo and KAGRA Collaborations (2025). *GW231123: A Binary Black Hole Merger with Total Mass 190–265 M⊙*.
- LIGO Scientific Collaboration and Virgo Collaboration (2019). *GWTC-1: A Gravitational-Wave Transient Catalog of Compact Binary Mergers Observed by LIGO and Virgo during the First and Second Observing Runs*.
- LIGO Scientific and Virgo Collaborations (2021). *GWTC-2: Compact Binary Coalescences Observed by LIGO and Virgo during the First Half of the Third Observing Run*.
- LIGO Scientific, Virgo and KAGRA Collaborations (2023). *GWTC-3: Compact Binary Coalescences Observed by LIGO and Virgo during the Second Part of the Third Observing Run*.
- Woosley (2017). *Pulsational Pair-instability Supernovae*.
- Farmer et al. (2019). *Mind the Gap: The Location of the Lower Edge of the Pair-instability Supernova Black Hole Mass Gap*.
- Woosley and Heger (2021). *The Pair-instability Mass Gap for Black Holes*.